# Balance LLMs across GPUs

Decide which GPU serves each LLM so that no GPU is overloaded. A coding agent
rewrites a naive placement function; a fixed grader measures the busiest GPU on
50 workloads. Meta-Evolve gives the agent ten attempts and keeps the best.

These are the same cells as the [GPU placement docs page](./), which also explains
the recorded run. Use a Python 3.12+ kernel with Docker running, install the
[OpenCode](https://opencode.ai) CLI, and set `OPENROUTER_API_KEY` for the agent;
scoring needs neither the CLI nor the key. The saved outputs are from the recorded run.

## Set up

Install the dependencies (the task files target this Meta-Evolve revision).

In [ ]:
%pip install -q numpy opencode-ai "meta-evolve @ git+https://github.com/sentient-xyz/meta-evolve.git@69354f9ebb66df37e5bfc78c26aecff2d1d78fae"

Download the two task files: the task, with its workloads, sandbox and grader
(`prism.py`), and the agent, with its instructions, permissions and check command
(`opencode_agent.py`).

In [ ]:
from urllib.request import urlretrieve

BASE = "https://sentient-xyz.github.io/meta-evolve-docs/applications/prism/"
for name in ["prism.py", "opencode_agent.py"]:
    urlretrieve(BASE + name, name)

A GPU's KV-cache pressure grows with the load on it and shrinks with the memory it
has left. **Lower is better.**

```text
pressure(gpu)  = sum(req_rate / slo of its models) / (80 GB - memory they use)
mean_max_kvpr  = the busiest GPU's pressure, averaged over 50 workloads
```

## 1. Load the seed

The seed is the naive first-fit placement from the ADRS PRISM task. It packs models
by size and never looks at load.

In [ ]:
import meta_evolve as meta
from prism import REFERENCES, SEED_SOURCE, DockerSandbox, evaluate

print(SEED_SOURCE)

def compute_model_placement(gpu_num, models):
    placement = {gpu_id: [] for gpu_id in range(gpu_num)}
    for model in models:
        for gpu_id in range(gpu_num):
            used = sum(item.model_size for item in placement[gpu_id])
            if model.model_size <= 80 - used:
                placement[gpu_id].append(model)
                break
    return placement



## 2. Score a candidate

Each candidate runs in a container with no network, no host files and no API key;
the grader checks its placements outside the container. Seeing these three numbers
means Docker and the grader work.

In [ ]:
sandbox = DockerSandbox()

def score(source):
    return evaluate(source, sandbox=sandbox)

for name, source in REFERENCES.items():
    print(f"{name:<20} {score(source).metrics['mean_max_kvpr']:.6f}")

first_fit_seed       320001.131240
prism_greedy         0.047866
adrs_released_best   0.040456


## 3. Propose with a coding agent

On each trial the OpenCode agent edits `candidate.py`, scores it with `prism-check`
(its only allowed command), repairs it for up to 15 steps, and returns the best
version that passed. Meta-Evolve then scores the proposal itself.

In [ ]:
from opencode_agent import OpenCodeAgent

agent = OpenCodeAgent("prism_run/agent")

## 4. Try ten trials

`TreeSearch` picks which program to improve on each trial; `AncestorsOnly` and
`PullAccess` give the agent the earlier trials' ideas and scores. Expect real model usage.

In [ ]:
from meta_evolve.policies import TreeSearch

experiment = meta.Experiment(
    task=meta.Task(
        evaluator=score,
        artifact=meta.Text,
        objectives=(meta.Minimize("mean_max_kvpr"),),
        budget=meta.Budget(trials=10, spend_micros=2_000_000),  # stop at 10 trials or $2
    ),
    seed=meta.Text(SEED_SOURCE),
    proposer=agent,
    search=TreeSearch(max_trials=10),
    context=meta.AncestorsOnly(max_records=40, max_chars=12_000),
    experience=meta.PullAccess(max_operations=1, max_results=12, max_records=24, max_chars=12_000),
    random_seed=20260921,
)
with agent:
    result = meta.run(experiment)

for trial in result.trials():
    kvpr = trial.metrics.get("mean_max_kvpr")
    print(f"trial {trial.logical_step}: " + (f"{kvpr:.6f}" if kvpr is not None else trial.failure.kind))

trial 0: 320001.131240
trial 1: 0.040412
trial 2: 0.039677
trial 3: 0.039595
trial 4: 0.039709
trial 5: 0.039730
trial 6: 0.039637
trial 7: 0.039630
trial 8: 0.039595
trial 9: 0.040011
trial 10: 0.042479


## 5. Check it on unseen workloads

The search only saw the 50 training workloads. Score the selected program once on
100 new ones.

In [ ]:
from prism import HELD_OUT_CASES, HELD_OUT_SEED

rivals = {"prism_greedy": REFERENCES["prism_greedy"],
          "adrs_released_best": REFERENCES["adrs_released_best"],
          "meta_evolve": result.best().value}
for name, source in rivals.items():
    metrics = evaluate(source, sandbox=sandbox, seed=HELD_OUT_SEED, num_tests=HELD_OUT_CASES).metrics
    print(f"{name:<20} {metrics['mean_max_kvpr']:.6f}  score {metrics['combined_score']:.2f}")

prism_greedy         0.052138  score 20.18
adrs_released_best   0.042716  score 24.41
meta_evolve          0.041890  score 24.87


## How to read this

Trial 3 was selected. On the unseen workloads it keeps the busiest GPU 19.7% less
loaded than PRISM greedy and 1.9% less than the best released ADRS program. The
docs page shows the search tree and what the agent tried on each trial:
[The recorded run](./#the-recorded-run).

To try another model, search method, memory or objective, see
[Try your own placement](./#try-your-own-placement).

Task: [UCB-ADRS/ADRS PRISM](https://github.com/UCB-ADRS/ADRS/tree/main/openevolve/examples/ADRS/prism) ·
[arXiv 2510.06189](https://arxiv.org/abs/2510.06189)